# Echo — Groupe 6

Faut-il généraliser les notifications agressives chez Echo ?

## Inspection des données

Les contrôles ci-dessous portent sur les CSV bruts. Aucune correction n'est appliquée.

In [1]:
from pathlib import Path
import pandas as pd

# Le notebook peut être ouvert depuis la racine du dépôt.
data_dir = Path("data")
if not data_dir.is_dir():
    raise FileNotFoundError("Ouvrir le notebook depuis le dossier GrowthModel.")

users_raw = pd.read_csv(data_dir / "users.csv", dtype={"user_id": "string"})
activity_raw = pd.read_csv(data_dir / "weekly_activity.csv", dtype={"user_id": "string"})
spend_raw = pd.read_csv(data_dir / "marketing_spend.csv")
tables = {"users": users_raw, "weekly_activity": activity_raw, "marketing_spend": spend_raw}

for name, table in tables.items():
    print(f"{name} : {len(table):,} lignes, {len(table.columns)} colonnes")
    print(table.dtypes.to_string())
    print()


users : 20,909 lignes, 7 colonnes
user_id                 string[python]
signup_date                     object
acquisition_channel             object
platform                        object
age_bucket                      object
push_variant                    object
shows_followed_week1             int64

weekly_activity : 334,399 lignes, 6 colonnes
user_id                        string[python]
week_start                             object
sessions                                int64
listening_minutes                     float64
ad_impressions                          int64
push_notifications_received             int64

marketing_spend : 17 lignes, 3 colonnes
month         object
channel       object
spend_eur    float64



### Valeurs manquantes et doublons

In [2]:
keys = {"users": ["user_id"], "weekly_activity": ["user_id", "week_start"], "marketing_spend": ["month", "channel"]}
audit_rows = []
for name, table in tables.items():
    distinct_rows = table.drop_duplicates()
    audit_rows.append({
        "table": name,
        "lignes": len(table),
        "copies_exactes": int(table.duplicated().sum()),
        "cles_repetees": int(table.duplicated(keys[name]).sum()),
        "lignes_en_conflit_apres_dedoublonnage": int(distinct_rows.duplicated(keys[name], keep=False).sum()),
    })
    print(f"Valeurs manquantes — {name}")
    print(table.isna().sum().to_string())
    print()
print(pd.DataFrame(audit_rows).to_string(index=False))


Valeurs manquantes — users
user_id                   0
signup_date               0
acquisition_channel       0
platform                  0
age_bucket              660
push_variant              0
shows_followed_week1      0

Valeurs manquantes — weekly_activity
user_id                        0
week_start                     0
sessions                       0
listening_minutes              0
ad_impressions                 0
push_notifications_received    0

Valeurs manquantes — marketing_spend
month        0
channel      0
spend_eur    0

          table  lignes  copies_exactes  cles_repetees  lignes_en_conflit_apres_dedoublonnage
          users   20909             124            124                                      0
weekly_activity  334399            1663           1663                                      0
marketing_spend      17               0              0                                      0


### Dates et catégories

Les dates avec `/` sont lues au format jour/mois/année. Les jours supérieurs à 12 permettent de vérifier cette convention. Les colonnes de contrôle ne remplacent pas les données brutes.

In [3]:
slash_dates = users_raw["signup_date"].str.contains("/", regex=False)
signup_dates = pd.to_datetime(users_raw["signup_date"], format="%Y-%m-%d", errors="coerce")
signup_dates.loc[slash_dates] = pd.to_datetime(users_raw.loc[slash_dates, "signup_date"], format="%d/%m/%Y", errors="coerce")
week_dates = pd.to_datetime(activity_raw["week_start"], format="%Y-%m-%d", errors="coerce")
month_dates = pd.to_datetime(spend_raw["month"], format="%Y-%m-%d", errors="coerce")

print("Dates d'inscription avec / :", int(slash_dates.sum()))
print("Dont jour supérieur à 12 :", int((users_raw.loc[slash_dates, "signup_date"].str[:2].astype(int) > 12).sum()))
print("Dates invalides (inscription, semaine, dépense) :", int(signup_dates.isna().sum()), int(week_dates.isna().sum()), int(month_dates.isna().sum()))
print("Inscriptions :", signup_dates.min().date(), "à", signup_dates.max().date())
print("Activité :", week_dates.min().date(), "à", week_dates.max().date())
print("Semaines ne commençant pas un lundi :", int((week_dates.dt.dayofweek != 0).sum()))
print("Dépenses non datées du premier du mois :", int((month_dates.dt.day != 1).sum()))
for column in ["acquisition_channel", "platform", "age_bucket", "push_variant"]:
    print(f"\n{column}")
    print(users_raw[column].value_counts(dropna=False).to_string())


Dates d'inscription avec / : 314
Dont jour supérieur à 12 : 193
Dates invalides (inscription, semaine, dépense) : 0 0 0
Inscriptions : 2025-01-01 à 2026-06-30
Activité : 2024-12-30 à 2026-06-29
Semaines ne commençant pas un lundi : 0
Dépenses non datées du premier du mois : 0

acquisition_channel
acquisition_channel
paid_social            8950
organic_store          7722
podcast_cross_promo    4237

platform
platform
Android    10496
iOS        10413

age_bucket
age_bucket
25-34    7043
18-24    5964
35-44    4143
45+      3099
NaN       660

push_variant
push_variant
standard           18181
control             1276
aggressive_push     1240
Control              100
AGGRESSIVE_PUSH       58
aggressive            54


### Valeurs numériques

In [4]:
print(activity_raw.select_dtypes("number").describe(percentiles=[0.5, 0.99]).round(2).to_string())
print("\nDurées négatives :", int((activity_raw["listening_minutes"] < 0).sum()))
# Une semaine entière représente 10 080 minutes.
print("Durées supérieures à une semaine :", int((activity_raw["listening_minutes"] > 7 * 24 * 60).sum()))
print("Émissions suivies en semaine 1 :")
print(users_raw["shows_followed_week1"].describe().to_string())
for name, table, columns in [
    ("users", users_raw, ["shows_followed_week1"]),
    ("weekly_activity", activity_raw, ["sessions", "ad_impressions", "push_notifications_received"]),
    ("marketing_spend", spend_raw, ["spend_eur"]),
]:
    print(name, "— valeurs négatives :", (table[columns] < 0).sum().to_dict())


        sessions  listening_minutes  ad_impressions  push_notifications_received
count  334399.00          334399.00       334399.00                    334399.00
mean        3.51             114.53           11.56                         5.26
std         1.84            1737.45            6.88                         6.23
min         1.00              -1.00            1.00                         0.00
50%         3.00              70.10           10.00                         4.00
99%         9.00             264.30           32.00                        36.00
max        21.00           99999.00           82.00                        58.00

Durées négatives : 96
Durées supérieures à une semaine : 100
Émissions suivies en semaine 1 :
count    20909.000000
mean         2.211010
std          1.551813
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         12.000000
users — valeurs négatives : {'shows_followed_week1': 0}
weekly_activity — valeurs

### Cohérence entre les fichiers et couverture

Une semaine sans ligne signifie qu'aucune activité n'est enregistrée. Elle ne prouve pas un départ définitif. La première et la dernière semaine peuvent être partielles.

In [5]:
print("Utilisateurs de l'activité absents de users :", len(set(activity_raw["user_id"]) - set(users_raw["user_id"])))
print("Utilisateurs sans activité :", len(set(users_raw["user_id"]) - set(activity_raw["user_id"])))

# Dédupliquer seulement la table de correspondance pour éviter une jointure plusieurs-à-plusieurs.
signup_lookup = pd.DataFrame({"user_id": users_raw["user_id"], "signup_date": signup_dates}).drop_duplicates()
joined = activity_raw[["user_id", "week_start"]].merge(signup_lookup, on="user_id", how="left", validate="many_to_one")
signup_monday = joined["signup_date"] - pd.to_timedelta(joined["signup_date"].dt.dayofweek, unit="D")
print("Activités antérieures à la semaine d'inscription :", int((pd.to_datetime(joined["week_start"]) < signup_monday).sum()))
print("Activités après l'extraction du 30/06/2026 :", int((week_dates > pd.Timestamp("2026-06-30")).sum()))

all_months = pd.date_range("2025-01-01", "2026-06-01", freq="MS")
for channel in spend_raw["channel"].unique():
    recorded_months = month_dates[spend_raw["channel"] == channel]
    print("Mois de dépenses absents pour", channel, ":", all_months.difference(recorded_months).strftime("%Y-%m").tolist())
print("Canaux sans dépenses renseignées :", sorted(set(users_raw["acquisition_channel"]) - set(spend_raw["channel"])))

test_users = pd.DataFrame({"signup_date": signup_dates, "push_variant": users_raw["push_variant"]}).loc[users_raw["push_variant"] != "standard"]
outside_test = ~test_users["signup_date"].between("2026-01-15", "2026-03-15")
print("Lignes de participants inscrits hors de la période du test :", int(outside_test.sum()))


Utilisateurs de l'activité absents de users : 0
Utilisateurs sans activité : 0
Activités antérieures à la semaine d'inscription : 0
Activités après l'extraction du 30/06/2026 : 0
Mois de dépenses absents pour paid_social : ['2025-05']
Canaux sans dépenses renseignées : ['organic_store', 'podcast_cross_promo']
Lignes de participants inscrits hors de la période du test : 0


### Bilan

- 124 copies exactes dans `users` et 1 663 dans `weekly_activity`, sans conflit de valeurs sur les clés.
- 314 dates d'inscription au format jour/mois/année.
- Variantes à harmoniser : `Control`, `AGGRESSIVE_PUSH` et `aggressive`.
- 660 âges absents dans le brut.
- 96 durées négatives et 100 durées de 99 999 minutes dans le brut.
- Dépense `paid_social` de mai 2025 absente. Aucun coût renseigné pour les autres canaux.
- Aucun indicateur fiable ne permet d'identifier des comptes internes ou de test. Aucun compte n'est exclu sur une simple suspicion.

Les règles validées sont appliquées dans la section suivante. Les effectifs changeront après retrait des copies exactes.

## Nettoyage

Choix validés : conserver les utilisateurs sans âge dans la catégorie `inconnu`, remplacer uniquement les durées impossibles par une valeur manquante, et estimer la dépense de mai 2025 entre avril et juin. Les CSV bruts restent intacts. Le journal compte les lignes affectées à chaque étape, dans l'ordre d'exécution.

In [6]:
users = users_raw.copy()
activity = activity_raw.copy()
spend = spend_raw.copy()
cleaning_log = []

def log_rule(table, rule, reason, affected, before, after):
    cleaning_log.append({
        "ordre": len(cleaning_log) + 1,
        "table": table,
        "regle": rule,
        "justification": reason,
        "lignes_affectees": int(affected),
        "lignes_avant": int(before),
        "lignes_apres": int(after),
    })

for name, table in [("users", users), ("weekly_activity", activity)]:
    before = len(table)
    table.drop_duplicates(inplace=True)
    log_rule(name, "Supprimer les copies exactes", "Une observation ne doit être comptée qu'une fois. Aucun conflit sur les clés après dédoublonnage.", before - len(table), before, len(table))

slash = users["signup_date"].str.contains("/", regex=False)
parsed_signup = pd.to_datetime(users["signup_date"], format="%Y-%m-%d", errors="coerce")
parsed_signup.loc[slash] = pd.to_datetime(users.loc[slash, "signup_date"], format="%d/%m/%Y", errors="raise")
assert parsed_signup.notna().all(), "Date d'inscription non reconnue"
users["signup_date"] = parsed_signup
activity["week_start"] = pd.to_datetime(activity["week_start"], format="%Y-%m-%d", errors="raise")
spend["month"] = pd.to_datetime(spend["month"], format="%Y-%m-%d", errors="raise")
log_rule("users", "Uniformiser les dates d'inscription en AAAA-MM-JJ", "Les dates avec / suivent le format jour/mois/année, confirmé par les jours supérieurs à 12.", slash.sum(), len(users), len(users))

variant_mapping = {"Control": "control", "AGGRESSIVE_PUSH": "aggressive_push", "aggressive": "aggressive_push"}
changed_variants = users["push_variant"].isin(variant_mapping)
users["push_variant"] = users["push_variant"].replace(variant_mapping)
log_rule("users", "Harmoniser les libellés des variantes push", "Les trois libellés correspondent aux variantes décrites dans le data pack.", changed_variants.sum(), len(users), len(users))

missing_age = users["age_bucket"].isna()
users["age_bucket"] = users["age_bucket"].fillna("inconnu")
log_rule("users", "Remplacer l'âge absent par inconnu", "Conserver les utilisateurs et ne pas inventer leur âge. Les comparaisons par âge présenteront inconnu séparément.", missing_age.sum(), len(users), len(users))
print("Utilisateurs après nettoyage :", len(users))
print(users["push_variant"].value_counts().to_string())


Utilisateurs après nettoyage : 20785
push_variant
standard           18076
control             1366
aggressive_push     1343


### Durées d'écoute

Une durée négative est invalide. Une durée supérieure aux 10 080 minutes d'une semaine est impossible. On conserve les autres mesures et on ajoute `listening_minutes_invalid` pour retrouver les lignes concernées. Ces durées ne doivent pas être interprétées comme zéro.

In [7]:
negative_minutes = activity["listening_minutes"] < 0
over_week_minutes = activity["listening_minutes"] > 7 * 24 * 60
activity["listening_minutes_invalid"] = negative_minutes | over_week_minutes
activity.loc[activity["listening_minutes_invalid"], "listening_minutes"] = float("nan")

log_rule("weekly_activity", "Mettre les durées négatives à manquant", "Garder les sessions, impressions et notifications de la ligne.", negative_minutes.sum(), len(activity), len(activity))
log_rule("weekly_activity", "Mettre les durées supérieures à 10 080 minutes à manquant", "La durée dépasse une semaine entière. Aucun plafonnement ni remplacement par zéro.", over_week_minutes.sum(), len(activity), len(activity))
print("Lignes d'activité conservées :", len(activity))
print("Durées remplacées par une valeur manquante :", int(activity["listening_minutes_invalid"].sum()))


Lignes d'activité conservées : 332736
Durées remplacées par une valeur manquante : 195


### Dépense manquante

La dépense `paid_social` de mai 2025 est estimée à la moyenne d'avril et de juin : `(1 080 + 1 150) / 2 = 1 115 €`. `spend_is_estimated` distingue cette ligne des observations. Les analyses qui l'utilisent, notamment le CAC et le backtest 2025, devront signaler cette estimation. Pour un backtest, cette interpolation ne doit pas servir à prétendre prévoir mai avant de connaître juin. Aucun coût nul n'est supposé pour les autres canaux.

In [8]:
spend["spend_is_estimated"] = False
before = len(spend)
may = pd.Timestamp("2025-05-01")
assert not ((spend["month"] == may) & (spend["channel"] == "paid_social")).any()
neighbors = spend.loc[(spend["channel"] == "paid_social") & spend["month"].isin([pd.Timestamp("2025-04-01"), pd.Timestamp("2025-06-01")]), "spend_eur"]
assert len(neighbors) == 2 and neighbors.notna().all()
estimated_spend = neighbors.mean()
spend = pd.concat([spend, pd.DataFrame([{
    "month": may, "channel": "paid_social", "spend_eur": estimated_spend, "spend_is_estimated": True,
}])], ignore_index=True).sort_values(["month", "channel"]).reset_index(drop=True)
log_rule("marketing_spend", "Ajouter mai 2025 : 1 115 EUR estimés", "Moyenne des dépenses observées d'avril et juin. Ligne identifiée par spend_is_estimated ; montant non observé.", 1, before, len(spend))
print(spend.loc[spend["month"].between("2025-04-01", "2025-06-01")].to_string(index=False))


     month     channel  spend_eur  spend_is_estimated
2025-04-01 paid_social     1080.0               False
2025-05-01 paid_social     1115.0                True
2025-06-01 paid_social     1150.0               False


### Vérification et export

Les fichiers nettoyés sont enregistrés dans `data/clean/`. Le journal est dans `docs/cleaning_log.csv`. Aucun compte n'est supprimé sans indicateur fiable. La semaine du 29 juin 2026 est partielle et devra être traitée comme telle dans les analyses.

In [9]:
# Les clés sont uniques et tous les utilisateurs de l'activité sont connus.
assert users["user_id"].is_unique
assert not activity.duplicated(["user_id", "week_start"]).any()
assert not spend.duplicated(["month", "channel"]).any()
assert activity["user_id"].isin(users["user_id"]).all()
assert users["signup_date"].between("2025-01-01", "2026-06-30").all()
assert users["push_variant"].isin(["standard", "control", "aggressive_push"]).all()
assert users["age_bucket"].isin(["18-24", "25-34", "35-44", "45+", "inconnu"]).all()
assert activity["week_start"].dt.dayofweek.eq(0).all()
assert activity["listening_minutes"].dropna().between(0, 10080).all()
assert activity["listening_minutes"].isna().equals(activity["listening_minutes_invalid"])

# Le nettoyage des durées ne modifie pas les autres mesures.
activity_distinct = activity_raw.drop_duplicates()
for column in ["sessions", "ad_impressions", "push_notifications_received"]:
    assert activity[column].equals(activity_distinct[column])
assert len(activity) == len(activity_distinct)
assert len(users) == users_raw["user_id"].nunique()
assert spend["spend_is_estimated"].sum() == 1
assert estimated_spend == 1115
assert spend.loc[~spend["spend_is_estimated"], "spend_eur"].sum() == spend_raw["spend_eur"].sum()

log = pd.DataFrame(cleaning_log)
assert len(users_raw) - len(users) == log.loc[log["table"].eq("users"), "lignes_avant"].iloc[0] - log.loc[log["table"].eq("users"), "lignes_apres"].iloc[-1]
clean_dir = data_dir / "clean"
clean_dir.mkdir(exist_ok=True)
Path("docs").mkdir(exist_ok=True)
for filename, table in [("users.csv", users), ("weekly_activity.csv", activity), ("marketing_spend.csv", spend)]:
    table.to_csv(clean_dir / filename, index=False, date_format="%Y-%m-%d", encoding="utf-8")
log.to_csv("docs/cleaning_log.csv", index=False, encoding="utf-8")
print(log[["ordre", "table", "regle", "lignes_affectees", "lignes_avant", "lignes_apres"]].to_string(index=False))
print("\nVérifications réussies. Fichiers nettoyés et journal enregistrés.")


 ordre           table                                                     regle  lignes_affectees  lignes_avant  lignes_apres
     1           users                              Supprimer les copies exactes               124         20909         20785
     2 weekly_activity                              Supprimer les copies exactes              1663        334399        332736
     3           users         Uniformiser les dates d'inscription en AAAA-MM-JJ               311         20785         20785
     4           users                Harmoniser les libellés des variantes push               211         20785         20785
     5           users                        Remplacer l'âge absent par inconnu               657         20785         20785
     6 weekly_activity                    Mettre les durées négatives à manquant                95        332736        332736
     7 weekly_activity Mettre les durées supérieures à 10 080 minutes à manquant               100        33273